# 2006 Local Government Election Data Cleaning

This notebook prepares the 2006 local government election results for analysis. It loads the source CSV, standardizes column names, selects the required fields, converts text values into numeric types, removes excluded ballot records, aggregates results, calculates municipality-level turnout, and exports the cleaned data.

## Import pandas

`pandas` supplies the DataFrame operations used to inspect, transform, group, and export the election data.

In [1]:
# Import pandas for tabular data cleaning and aggregation.
import pandas as pd
from pathlib import Path

# Paths are relative to the project folder, so this notebook runs on any computer.
# It works whether Jupyter is opened in the notebooks/ folder or in the project root.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
CLEAN_DIR = PROJECT_DIR / "data" / "cleaned"

## Load the raw 2006 data

Read the source CSV from `data/raw/` using `ISO-8859-1` encoding so that accented or special characters in the election data are handled correctly. Display the DataFrame to inspect the imported records.

In [2]:
# Load the raw election results with an encoding that supports special characters.
df = pd.read_csv(RAW_DIR / "2006_LGE.csv.zip", encoding="ISO-8859-1")

# Display the imported data for an initial review.
df

/tmp/ipykernel_154/1927548902.py:2: DtypeWarning: Columns (0: Spoilt
Votes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_DIR / "2006_LGE.csv.zip", encoding="ISO-8859-1")


,Electoral Event,Province,Municipality,Ward,Voting \nDistrict,Party,Ballot \nType,Registered\nVoters,% Voter \nTurnout,MEC7\nVotes,Total Votes \nCast,Valid Votes \nCast,Spoilt\nVotes,Unnamed: 13
0,Local Government Elections 2006,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],20502001.0,11760993,AFRICAN NATIONAL CONGRESS,DC 40%,504,70.90%,14,358,333,4,NaN
1,Local Government Elections 2006,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],20502001.0,11760993,AFRICAN NATIONAL CONGRESS,PR,504,70.90%,14,356,331,3,NaN
2,Local Government Elections 2006,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],20502001.0,11760993,AFRICAN NATIONAL CONGRESS,WARD,504,70.90%,14,358,331,5,NaN
3,Local Government Elections 2006,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],20502001.0,11760993,INKATHA FREEDOM PARTY,DC 40%,504,70.90%,14,358,10,4,NaN
4,Local Government Elections 2006,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],20502001.0,11760993,INKATHA FREEDOM PARTY,PR,504,70.90%,14,356,10,3,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374690,Local Government Elections 2006,Western Cape,WCDMA05 [Central Karoo DC],NaN,98200032,INDEPENDENT CIVIC ORGANISATION OF SOUTH AFRICA,DC 40%,177,66.67%,0,118,22,6,NaN
374691,Local Government Elections 2006,Western Cape,WCDMA05 [Central Karoo DC],NaN,98200032,INDEPENDENT CIVIC ORGANISATION OF SOUTH AFRICA,DMA DC 60%,177,66.67%,0,118,24,4,NaN
374692,Local Government Elections 2006,Western Cape,WCDMA05 [Central Karoo DC],NaN,98200032,INDEPENDENT DEMOCRATS,DC 40%,177,66.67%,0,118,2,6,NaN
374693,Local Government Elections 2006,Western Cape,WCDMA05 [Central Karoo DC],NaN,98200032,INDEPENDENT DEMOCRATS,DMA DC 60%,177,66.67%,0,118,1,4,NaN


## Inspect the source columns

Review the original column names before cleaning them. This identifies formatting issues and confirms which fields are available.

In [3]:
# Inspect the original column names before standardizing them.
df.columns

Index(['Electoral Event', 'Province', 'Municipality', 'Ward',
       'Voting \nDistrict', 'Party', 'Ballot \nType', 'Registered\nVoters',
       '% Voter \nTurnout', 'MEC7\nVotes', 'Total Votes \nCast',
       'Valid Votes \nCast', 'Spoilt\nVotes', 'Unnamed: 13'],
      dtype='str')

## Clean column names

Remove embedded newline characters from every column name so the fields can be referenced consistently in later operations.

In [4]:
# Remove newline characters embedded in the source column labels.
df = df.rename(columns=lambda c: c.replace('\n', ''))

# Confirm the cleaned column names.
df.columns

Index(['Electoral Event', 'Province', 'Municipality', 'Ward',
       'Voting District', 'Party', 'Ballot Type', 'RegisteredVoters',
       '% Voter Turnout', 'MEC7Votes', 'Total Votes Cast', 'Valid Votes Cast',
       'SpoiltVotes', 'Unnamed: 13'],
      dtype='str')

## Select the analysis fields

Create a smaller working DataFrame containing the geographic identifiers, party, ballot type, turnout, and valid vote count needed for the analysis.

In [5]:
# Keep only the fields needed for the cleaning and aggregation steps.
draft = df.filter(axis=1, items=['Province','Municipality','Party','Ballot Type','% Voter Turnout','Valid Votes Cast'])

## Inspect the working DataFrame

Check the selected columns, data types, and non-null counts before converting the text-based numeric fields.

In [6]:
# Check the selected fields and their current data types.
draft.info()

<class 'pandas.DataFrame'>
RangeIndex: 374695 entries, 0 to 374694
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Province          374695 non-null  str  
 1   Municipality      374695 non-null  str  
 2   Party             374695 non-null  str  
 3   Ballot Type       374695 non-null  str  
 4   % Voter Turnout   374695 non-null  str  
 5   Valid Votes Cast  374695 non-null  str  
dtypes: str(6)
memory usage: 17.2 MB


## Convert turnout and vote totals to numbers

Remove the percent sign from turnout values and convert them to floating-point numbers. Remove thousands separators from valid vote totals and convert them to integers.


Display the converted DataFrame to confirm that the values are ready for aggregation.

In [7]:
# Remove the percent sign and convert turnout to a numeric percentage.
draft['% Voter Turnout'] = draft['% Voter Turnout'].str.removesuffix('%').astype('float')

# Remove thousands separators and convert vote totals to integers.
draft['Valid Votes Cast'] = draft['Valid Votes Cast'].str.replace(',',"", regex=False).astype('int')

# Display the converted values before filtering and aggregation.
draft

,Province,Municipality,Party,Ballot Type,% Voter Turnout,Valid Votes Cast
0,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],AFRICAN NATIONAL CONGRESS,DC 40%,70.90,333
1,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],AFRICAN NATIONAL CONGRESS,PR,70.90,331
2,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],AFRICAN NATIONAL CONGRESS,WARD,70.90,331
3,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],INKATHA FREEDOM PARTY,DC 40%,70.90,10
4,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],INKATHA FREEDOM PARTY,PR,70.90,10
...,...,...,...,...,...,...
374690,Western Cape,WCDMA05 [Central Karoo DC],INDEPENDENT CIVIC ORGANISATION OF SOUTH AFRICA,DC 40%,66.67,22
374691,Western Cape,WCDMA05 [Central Karoo DC],INDEPENDENT CIVIC ORGANISATION OF SOUTH AFRICA,DMA DC 60%,66.67,24
374692,Western Cape,WCDMA05 [Central Karoo DC],INDEPENDENT DEMOCRATS,DC 40%,66.67,2
374693,Western Cape,WCDMA05 [Central Karoo DC],INDEPENDENT DEMOCRATS,DMA DC 60%,66.67,1


## Check for duplicate rows

Inspect duplicate records without assigning the de-duplicated result back to `draft`. The second display shows the current working DataFrame.

In [8]:
# Inspect duplicates without assigning the result back to draft.
draft.drop_duplicates()

# Display the current working DataFrame.
draft

,Province,Municipality,Party,Ballot Type,% Voter Turnout,Valid Votes Cast
0,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],AFRICAN NATIONAL CONGRESS,DC 40%,70.90,333
1,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],AFRICAN NATIONAL CONGRESS,PR,70.90,331
2,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],AFRICAN NATIONAL CONGRESS,WARD,70.90,331
3,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],INKATHA FREEDOM PARTY,DC 40%,70.90,10
4,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],INKATHA FREEDOM PARTY,PR,70.90,10
...,...,...,...,...,...,...
374690,Western Cape,WCDMA05 [Central Karoo DC],INDEPENDENT CIVIC ORGANISATION OF SOUTH AFRICA,DC 40%,66.67,22
374691,Western Cape,WCDMA05 [Central Karoo DC],INDEPENDENT CIVIC ORGANISATION OF SOUTH AFRICA,DMA DC 60%,66.67,24
374692,Western Cape,WCDMA05 [Central Karoo DC],INDEPENDENT DEMOCRATS,DC 40%,66.67,2
374693,Western Cape,WCDMA05 [Central Karoo DC],INDEPENDENT DEMOCRATS,DMA DC 60%,66.67,1


## Exclude DC 40% records

Remove `DC 40%` ballot records before calculating grouped totals and mean turnout, keeping the analysis focused on the intended ballot categories.

In [9]:
# Exclude DC 40% ballot records from the analysis.
draft = draft[draft['Ballot Type']!="DC 40%"]

# Review the filtered records.
draft

,Province,Municipality,Party,Ballot Type,% Voter Turnout,Valid Votes Cast
1,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],AFRICAN NATIONAL CONGRESS,PR,70.90,331
2,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],AFRICAN NATIONAL CONGRESS,WARD,70.90,331
4,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],INKATHA FREEDOM PARTY,PR,70.90,10
5,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],INKATHA FREEDOM PARTY,WARD,70.90,12
7,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],PAN AFRICANIST CONGRESS OF AZANIA,PR,70.90,6
...,...,...,...,...,...,...
374684,Western Cape,WCDMA05 [Central Karoo DC],INDEPENDENT DEMOCRATS,DMA DC 60%,40.22,1
374687,Western Cape,WCDMA05 [Central Karoo DC],AFRICAN NATIONAL CONGRESS,DMA DC 60%,66.67,49
374689,Western Cape,WCDMA05 [Central Karoo DC],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,DMA DC 60%,66.67,40
374691,Western Cape,WCDMA05 [Central Karoo DC],INDEPENDENT CIVIC ORGANISATION OF SOUTH AFRICA,DMA DC 60%,66.67,24


## Aggregate records by party and ballot type

Group the filtered records by province, municipality, party, and ballot type. Sum valid votes and calculate the mean turnout across the ward-level records in each group.

In [10]:
# Aggregate vote totals and turnout by geography, party, and ballot type.
draft = (
    draft
    .groupby(['Province', 'Municipality','Party','Ballot Type'], as_index=False)
    .agg(
        ValidVotesCast=('Valid Votes Cast', 'sum'),
        MeanVoterTurnout=('% Voter Turnout', 'mean') # Turnout from each ward
    )
)

# This display shows the grouped records before adding municipality-level turnout.
# draft = draft[draft['Ballot Type']!="DC 40%"]
draft

,Province,Municipality,Party,Ballot Type,ValidVotesCast,MeanVoterTurnout
0,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],AFRICAN NATIONAL CONGRESS,PR,42012,58.521768
1,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],AFRICAN NATIONAL CONGRESS,WARD,41683,58.521768
2,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],INDEPENDENT,WARD,513,55.775806
3,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],INKATHA FREEDOM PARTY,PR,430,58.521768
4,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],INKATHA FREEDOM PARTY,WARD,252,55.589143
...,...,...,...,...,...,...
2926,Western Cape,WCDMA04 [South Cape DC],VRYHEIDSFRONT PLUS,DMA DC 60%,48,50.774444
2927,Western Cape,WCDMA05 [Central Karoo DC],AFRICAN NATIONAL CONGRESS,DMA DC 60%,1108,54.390000
2928,Western Cape,WCDMA05 [Central Karoo DC],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,DMA DC 60%,293,54.390000
2929,Western Cape,WCDMA05 [Central Karoo DC],INDEPENDENT CIVIC ORGANISATION OF SOUTH AFRICA,DMA DC 60%,351,54.390000


## Calculate municipality-level turnout

Average the grouped turnout values within each municipality to create one municipality-level turnout measure. Round the result to two decimal places for reporting.

In [11]:
# Average grouped turnout values to obtain one value per municipality.
municipality_turnout = (
    draft
    .groupby(['Municipality'], as_index=False)['MeanVoterTurnout']
    .mean()) # Turnout from each municipality

# Round municipality turnout percentages for consistent reporting.
municipality_turnout['MeanVoterTurnout'] = municipality_turnout['MeanVoterTurnout'].round(2)

# Review the municipality-level lookup table.
municipality_turnout

,Municipality,MeanVoterTurnout
0,CPT - City of Cape Town [Cape Town],50.24
1,EC05b2 - Umzimvubu [Mount Ayliff],57.89
2,EC05b3 - Matatiele [Matatiele],59.77
3,EC101 - Camdeboo [Graaff-Reinet],48.54
4,EC102 - Blue Crane Route [Somerset East],49.09
...,...,...
251,WCDMA01 [West Coast DC],53.68
252,WCDMA02 [Brede River DC],35.17
253,WCDMA03 [Overberg DC],44.79
254,WCDMA04 [South Cape DC],50.77


## Attach municipality turnout to each record

Remove the intermediate grouped turnout value and merge the municipality-level turnout lookup back into the aggregated records. This gives every party and ballot-type row its municipality's mean turnout.

In [12]:
# Replace grouped turnout with the municipality-level mean turnout.
draft = draft.drop(columns=['MeanVoterTurnout']).merge(
    municipality_turnout,
    on='Municipality',
    how='left'
 )

# Display the final cleaned records before export.
draft

,Province,Municipality,Party,Ballot Type,ValidVotesCast,MeanVoterTurnout
0,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],AFRICAN NATIONAL CONGRESS,PR,42012,57.89
1,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],AFRICAN NATIONAL CONGRESS,WARD,41683,57.89
2,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],INDEPENDENT,WARD,513,57.89
3,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],INKATHA FREEDOM PARTY,PR,430,57.89
4,Eastern Cape,EC05b2 - Umzimvubu [Mount Ayliff],INKATHA FREEDOM PARTY,WARD,252,57.89
...,...,...,...,...,...,...
2926,Western Cape,WCDMA04 [South Cape DC],VRYHEIDSFRONT PLUS,DMA DC 60%,48,50.77
2927,Western Cape,WCDMA05 [Central Karoo DC],AFRICAN NATIONAL CONGRESS,DMA DC 60%,1108,54.39
2928,Western Cape,WCDMA05 [Central Karoo DC],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,DMA DC 60%,293,54.39
2929,Western Cape,WCDMA05 [Central Karoo DC],INDEPENDENT CIVIC ORGANISATION OF SOUTH AFRICA,DMA DC 60%,351,54.39


## Export the cleaned dataset

Save the completed 2006 election dataset as `data/cleaned/2006_LGE_Cleaned.csv` for downstream analysis.

In [13]:
# Export the cleaned 2006 election data for downstream analysis.
draft.to_csv(CLEAN_DIR / '2006_LGE_Cleaned.csv')